# Xuan Thuy Prithvi-UNet: train, predict, compare

Runs in three places without edits:

| Where | How to start | Where the code runs | Data |
| --- | --- | --- | --- |
| Colab in the browser | open the notebook, *Runtime > Change runtime type > GPU* | Colab VM | Google Drive |
| VS Code + **Google Colab** extension | *Select Kernel > Colab > New Colab Server* (pick a GPU) | Colab VM, not your PC | Google Drive (`drive.mount` needs extension >= 0.2.1) or *Upload to Colab* |
| VS Code, local Python kernel | *Select Kernel > Python Environments* (venv with `pip install -e .`) | your PC | local folders |

With the Colab extension the notebook file is local but every cell runs on Google's VM: `E:\...` paths do not exist there, and Colab *Secrets* (`userdata.get`) are not available, so the GitHub token is typed when asked.

In [ ]:
import os, sys, glob, subprocess, pathlib
ON_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
print("Colab VM" if ON_COLAB else "local kernel", "| python", sys.version.split()[0])
try:
    import torch
    print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except ImportError:
    print("torch not installed yet")

## 1. Paths (edit once)

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                        # in VS Code also: command 'Colab: Mount Google Drive to Server...'
    BASE = "/content/drive/MyDrive/XuanThuy"             # your Drive folder
    REPO = "/content/xuanthuy-prithvi-unet"
else:
    BASE = r"E:\Learning things\Aetosky\XuanthuyPrithvi"   # your local folder
    REPO = os.path.join(BASE, "xuanthuy-prithvi-unet")

DATA_ROOT = os.path.join(BASE, "data")                   # GeoTIFFs named in the config + rasterized_vector_categorized.tif
RUNS = os.path.join(BASE, "runs")                        # one sub-folder per experiment
WEIGHTS = os.path.join(BASE, "weights", "Prithvi_EO_V2_300M_TL.pt")
CONFIG = "configs/p3_single_date.json"
print(DATA_ROOT, os.path.isdir(DATA_ROOT))

## 2. Code (private repository: a GitHub token with read access is asked once per VM)

In [ ]:
URL = "github.com/hongphuc-maker/xuanthuy-prithvi-unet.git"
if ON_COLAB and not os.path.isdir(REPO):
    import getpass
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "-q", f"https://x-access-token:{token}@{URL}", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://{URL}"], check=True)  # token not kept
    del token
os.chdir(REPO)
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[export,notebook]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Every stage, its output shape and its learned parameters (random weights, no download)
!python scripts/trace_shapes.py --config {CONFIG}

## 3. Prithvi weights: download once to Drive / local folder, reuse afterwards

In [ ]:
if not os.path.exists(WEIGHTS):
    from huggingface_hub import hf_hub_download
    hf_hub_download("ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL", "Prithvi_EO_V2_300M_TL.pt",
                    revision="63adbd39c271da4c42f447e69b1a7c91a338cdc9", local_dir=os.path.dirname(WEIGHTS))
os.environ["PRITHVI_WEIGHTS"] = WEIGHTS
print(WEIGHTS, round(os.path.getsize(WEIGHTS) / 1e9, 2), "GB")

## 4. Train (GPU). Interrupted runs resume from `latest_resume.pt` when the same cell is run again.

In [ ]:
RUN_DIR = os.path.join(RUNS, "p3_v1")                    # new folder for every config change
!python scripts/train.py --config {CONFIG} --data-root "{DATA_ROOT}" --run-dir "{RUN_DIR}" --steps 2   # quick check
# !python scripts/train.py --config {CONFIG} --data-root "{DATA_ROOT}" --run-dir "{RUN_DIR}"            # full run

## 5. Map a new date or a new area (CPU is enough)

In [ ]:
CHECKPOINT = os.path.join(RUNS, "p3_v1", "best_any.pt")  # a legacy best_any.pt of the old repo also works
NEW_IMAGES = sorted(glob.glob(os.path.join(DATA_ROOT, "Sentinel2_2026-10-01_*.tif")))   # P3: 1 file, P2: 6 files
OUT = os.path.join(BASE, "maps", "p3_20261001")
IMGS = " ".join(f'"{p}"' for p in NEW_IMAGES)      # quoted paths for the shell
!python scripts/band_statistics.py {IMGS} --offset auto
!python scripts/predict.py --config {CONFIG} --checkpoint "{CHECKPOINT}" --images {IMGS} --out "{OUT}" --dn-offset auto

## 6. Does the final bilinear x4 smooth the map? Same logits, nearest instead of bilinear

In [ ]:
import numpy as np, rasterio, matplotlib.pyplot as plt
OUT_NEAREST = OUT + "_nearest"
!python scripts/predict.py --config {CONFIG} --checkpoint "{CHECKPOINT}" --images {IMGS} --out "{OUT_NEAREST}" --dn-offset auto --upsample nearest
a = rasterio.open(glob.glob(os.path.join(OUT, "*_classes.tif"))[0]).read(1)
b = rasterio.open(glob.glob(os.path.join(OUT_NEAREST, "*_classes.tif"))[0]).read(1)
valid = a > 0
print(f"pixels whose class changes: {100 * (a[valid] != b[valid]).mean():.2f} %")
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
h, w = a.shape
r, c = slice(h // 2 - 150, h // 2 + 150), slice(w // 2 - 150, w // 2 + 150)   # central 3 x 3 km
ax[0].imshow(a[r, c], cmap="tab20", vmin=0, vmax=11); ax[0].set_title("bilinear (as trained)")
ax[1].imshow(b[r, c], cmap="tab20", vmin=0, vmax=11); ax[1].set_title("nearest")
ax[2].imshow((a != b)[r, c], cmap="gray"); ax[2].set_title("changed pixels")
for x in ax: x.axis("off")

## 7. Optional: export images for another area or period (Planetary Computer)

In [ ]:
AOI = os.path.join(BASE, "aoi", "my_area.geojson")
EXPORT_OUT = os.path.join(DATA_ROOT, "my_area")
!python scripts/export_sentinel2.py --aoi "{AOI}" --start 2026-10-01 --end 2026-10-31 --out "{EXPORT_OUT}" --dry-run